# Ignitions → ArcGIS Online (publish)

Run in **ArcGIS Pro on the VM** (signed in to ArcGIS Online) — like pods-atlas `01AGOL_Prep.ipynb`, this notebook needs only Pro's own Python (`arcgis`), not `fb_tools`. Build the GeoPackage first on the Mac with `IgnitionAGOL.ipynb` §1–3.

## Publish options

**First time (browser, a few minutes):** ArcGIS Online → *Content* → *New item* → drop the GeoPackage →
*Add and create a hosted feature layer*. One item holds all five layers. Then in a web map:

- `ignitions`: *Counts and Amounts (size)* on `acres`, or *Unique symbols* on `size_class`; filter on `year`.
- `density_human`, `density_lightning`: *Unique symbols* on `class_id` (light → dark), pop-up `{label}`.
- `share_human`: *Unique symbols* on `class_id`, blue → grey → orange.
- Add the streamed layers by URL (*Add* → *Add layer from URL*):
  - `https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/InterAgencyFirePerimeterHistory_All_Years_View/FeatureServer`
  - `https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Interagency_Perimeters/FeatureServer`
  - `https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Incident_Locations_Current/FeatureServer`

**Every later run:** on the hosted layer's item page, *Update data* → *Overwrite entire layer* with the new
GeoPackage (same file name). The item id and URL stay the same, so the web map keeps working. Overwrite needs the
same layers and fields — **new data goes in a new item** added to the same web map, not as a new layer here.

**From ArcGIS Pro (this notebook):** *Run All*. The first run uploads the GeoPackage and publishes one hosted feature layer; later runs find it by title and overwrite it in place. `FRESH = True` deletes and republishes (the pods-atlas route) if the overwrite fails or the layers/fields change.

In [2]:
# On the Mac (no arcpy) this only finds the GeoPackage; in ArcGIS Pro it publishes.
import importlib.util
from pathlib import Path

TITLE = "Larimer wildfire ignitions"
REL   = Path("data/spatial/mod/ignition_density/larimer/agol/larimer_ignitions_agol.gpkg")
ROOTS = [*(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "fb_tools").is_dir()),          # repo above cwd
         Path.home() / "Library/CloudStorage/Box-Box/MCC/fire_modeling/FM_PythonWrapper",          # Mac
         Path(r"\\Mac\Home\Library\CloudStorage\Box-Box\MCC\fire_modeling\FM_PythonWrapper")]      # Parallels VM
gpkg = next((r / REL for r in ROOTS if (r / REL).exists()), None)
assert gpkg is not None, f"{REL} not found under {[str(r) for r in ROOTS]} — run §1–3 on the Mac first"
print(gpkg)

if importlib.util.find_spec("arcpy") is None:
    print("Not in ArcGIS Pro — open this notebook in Pro on the VM and run this cell to publish.")
else:
    from arcgis.features import FeatureLayerCollection
    from arcgis.gis import GIS

    # Same calls as pods-atlas 01AGOL_Prep (GIS("pro"), root folder add → .result() → publish).
    FRESH = False   # True: delete the item and publish new (pods-atlas style) — use if overwrite fails or the
                    # layers/fields changed; the item id and URL change, so re-point the web map.
    gis = GIS("pro")                     # Pro's active sign-in
    me = gis.users.me
    print(f"Connected: {me.username} @ {gis.properties.portalHostname}")
    mine = [i for i in gis.content.search(f'title:"{TITLE}" AND owner:{me.username}', max_items=25)
            if i.title == TITLE]
    service = next((i for i in mine if i.type == "Feature Service"), None)

    if service is not None and not FRESH:     # later runs: overwrite in place — item id and URL stay the same
        print(FeatureLayerCollection.fromitem(service).manager.overwrite(str(gpkg)))
        item = service
    else:                                     # first run (or FRESH): upload the GeoPackage, publish one layer
        for it in mine:
            if it.type in ("GeoPackage", "Feature Service"):
                print(f"Deleting {it.type} {it.id}")
                try:
                    it.protect(enable=False)
                except Exception:
                    pass
                try:
                    it.delete()
                except Exception as e:        # a deleted source can take its service with it
                    print(f"  (skip: {e})")
        props = {"title": TITLE, "type": "GeoPackage", "tags": "wildfire, ignitions, FPA-FOD, Larimer",
                 "snippet": "Wildfire ignitions 2013–2024 and ignition density by cause, "
                            "Larimer County and neighboring counties."}
        src_item = gis.content.folders.get("/").add(props, file=str(gpkg)).result()
        item = src_item.publish()
    print(f"{gis.url.rstrip('/')}/home/item.html?id={item.id}\n{item.url}")
    for lyr in item.layers:
        print(f"  {lyr.properties.name}: {lyr.query(return_count_only=True):,} features")


\\Mac\Home\Library\CloudStorage\Box-Box\MCC\fire_modeling\FM_PythonWrapper\data\spatial\mod\ignition_density\larimer\agol\larimer_ignitions_agol.gpkg
Connected: C829470251_CSUesri @ www.arcgis.com
https://www.arcgis.com/home/item.html?id=791ec44e7d75470eaf71da975474031e
https://services1.arcgis.com/KNdRU5cN6ENqCTjk/arcgis/rest/services/Larimer_wildfire_ignitions/FeatureServer
  main.counties: 7 features
  main.density_human: 8 features
  main.density_lightning: 6 features
  main.ignitions: 11,617 features
  main.share_human: 10 features
